# LSTM for CMAPSS FD001 RUL Prediction (corrected v2)

## What changed from `04_LSTM_corrected_all_sensor.ipynb`, and why

Your last run reported **R² = 0.9044, MAE 9.43, RMSE 12.94** on the internal
20-engine validation split, with train MAE 10.01 vs val MAE 9.70 (val
*better* than train -- not overfitting). Three things, in priority order:

1. **Fixed a real bug**: the previous notebook's own data-validation cell
   printed `"WARNING: Expected 18 features... but found 24"` and then
   *kept running anyway*. The file paths pointed at
   `(not_removed_the_sensor)` despite the markdown claiming the switch to
   the cleaned 18-feature data. That warning is now a hard `assert` that
   stops execution instead of silently continuing on the wrong data.
2. **Added trend features** (rolling 5-cycle mean + rate-of-change per
   sensor, via `create_sequence_corrected.py`). Since the train/val gap
   showed no overfitting, the checklist says look at data/features next,
   not architecture -- this gives the LSTM an explicit degradation-slope
   signal. Feature count goes from 18 to 54 (original + roll_mean + roc).
   Expected effect: a modest real gain, not a jump to 0.99 -- published
   FD001 benchmarks sit around R² 0.85-0.92, so 0.90 was already near the
   top of that range.
3. **Added two evaluations that didn't exist before**, both aimed at your
   actual goal (catching failure before it happens), not just R²:
   - Error specifically on `RUL <= 30` samples (the only region where
     "detect it before it happens" is actually being tested -- a good
     global R² can hide poor accuracy exactly here).
   - Evaluation on the **official NASA test set** (`test_FD001.txt` +
     `RUL_FD001.txt`), via `prepare_official_test_set.py`. Your existing
     validation split is 20 engines held out from the *same* 100 training
     engines; the official test set is truncated at arbitrary points and
     is the actual literature benchmark. This is the number to trust.

Architecture, RUL capping, splitting, and scaling are unchanged -- none of
them were the problem, and there's no signal here calling for more
capacity or more regularization.


In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

WINDOW_SIZE_EXPECTED = 30
NUMBER_OF_FEATURES_EXPECTED = 54  # 18 base + 18 roll_mean + 18 roc (constant sensors removed)
RUL_CAP = 125

## Data paths (FIXED)

These now correctly point at the `(removed_the_sensor)` sequence files
produced by `create_sequence_corrected.py` -- 18 base features plus the
new trend channels, RUL capped at 125. This is the actual bug fix: the
previous notebook loaded `(not_removed_the_sensor)` despite claiming
otherwise.


In [ ]:
X_TRAIN_PATH = (
    "../CMAPSSData/Processed/"
    "X_train_sequences(removed_the_sensor).npy"
)

Y_TRAIN_PATH = (
    "../CMAPSSData/Processed/"
    "y_train_sequences(removed_the_sensor).npy"
)

X_VAL_PATH = (
    "../CMAPSSData/Processed/"
    "X_val_sequences(removed_the_sensor).npy"
)

Y_VAL_PATH = (
    "../CMAPSSData/Processed/"
    "y_val_sequences(removed_the_sensor).npy"
)

# Official NASA held-out test set, produced by prepare_official_test_set.py
X_TEST_OFFICIAL_PATH = "../CMAPSSData/Processed/X_test_official.npy"
Y_TEST_OFFICIAL_PATH = "../CMAPSSData/Processed/y_test_official.npy"

In [ ]:
BEST_MODEL_PATH = (
    "../Models/Reduced_Sensor_Capped_Trend/"
    "LSTM_FD001_trend_best.keras"
)

FINAL_MODEL_PATH = (
    "../Models/Reduced_Sensor_Capped_Trend/"
    "LSTM_FD001_trend_final.keras"
)

HISTORY_PATH = (
    "../Models/Reduced_Sensor_Capped_Trend/"
    "LSTM_FD001_trend_history.npz"
)

In [ ]:
X_train = np.load(X_TRAIN_PATH)
y_train = np.load(Y_TRAIN_PATH)

X_val = np.load(X_VAL_PATH)
y_val = np.load(Y_VAL_PATH)

print("\nTraining data:")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nValidation data:")
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

In [ ]:
print("DATA VALIDATION")

number_of_features = X_train.shape[2]

# FIXED: this used to only print a warning and continue. Now it stops
# execution -- if this fires, re-run create_sequence_corrected.py and
# check you're pointing at the (removed_the_sensor) files.
assert number_of_features == NUMBER_OF_FEATURES_EXPECTED, (
    f"Expected {NUMBER_OF_FEATURES_EXPECTED} features (18 base + trend "
    f"channels, constant sensors removed), but found {number_of_features}. "
    f"Check X_TRAIN_PATH / X_VAL_PATH point at the (removed_the_sensor) "
    f"files produced by create_sequence_corrected.py, not "
    f"(not_removed_the_sensor)."
)

assert X_val.shape[2] == number_of_features, (
    "Training and validation datasets have different numbers of features."
)

assert len(X_train) == len(y_train), "X_train/y_train length mismatch."
assert len(X_val) == len(y_val), "X_val/y_val length mismatch."

assert y_train.max() <= RUL_CAP, (
    "y_train max exceeds RUL_CAP -- capping was not applied upstream."
)

assert not np.isnan(X_train).any(), "NaN values in X_train."
assert not np.isnan(y_train).any(), "NaN values in y_train."
assert not np.isnan(X_val).any(), "NaN values in X_val."
assert not np.isnan(y_val).any(), "NaN values in y_val."

assert not np.isinf(X_train).any(), "Infinite values in X_train."
assert not np.isinf(X_val).any(), "Infinite values in X_val."

print("All checks passed.")
print("Number of features:", number_of_features)
print("Window size:", X_train.shape[1])

## Model architecture (UNCHANGED)

Kept exactly as it was: three stacked `Bidirectional` LSTM layers +
`LSTM(32)`, `Dropout(0.2)` throughout, no `BatchNormalization`, no L2.
The train/val gap in the last run was **negative** (val MAE slightly
better than train MAE) -- that is not an overfitting signal, so per the
"don't add anti-overfitting techniques defensively" rule, capacity and
regularization are left alone here. The lever pulled this round is the
trend features, tested on their own against this same architecture.


In [ ]:
print("Building LSTM Model")

INPUT_SHAPE = X_train.shape[1:]

model = Sequential([
    Input(shape=INPUT_SHAPE),

    Bidirectional(
        LSTM(256, return_sequences=True)
    ),
    Dropout(0.2),

    Bidirectional(
        LSTM(128, return_sequences=True)
    ),
    Dropout(0.2),
    Bidirectional(
        LSTM(64, return_sequences=True)
    ),
    Dropout(0.2),

    LSTM(32, return_sequences=False),
    Dropout(0.2),

    Dense(16, activation="relu"),
    Dense(1)
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=12,
    mode='min',
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    mode='min',
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    BEST_MODEL_PATH,
    monitor="val_loss",
    mode="min",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stopping, reduce_lr, model_checkpoint],
    verbose=1
)

model.save(FINAL_MODEL_PATH)

np.savez(
    HISTORY_PATH,
    loss=np.array(history.history["loss"]),
    val_loss=np.array(history.history["val_loss"]),
    mae=np.array(history.history["mae"]),
    val_mae=np.array(history.history["val_mae"])
)

In [ ]:
final_train_mae = history.history['mae'][-1]
final_val_mae = history.history['val_mae'][-1]
gap = final_val_mae - final_train_mae

print("Actual epochs completed:", len(history.history['loss']))
print("Best validation loss:", min(history.history["val_loss"]))
print("Best validation MAE:", min(history.history["val_mae"]))
print(f"\nFinal train MAE: {final_train_mae:.3f}")
print(f"Final val MAE:   {final_val_mae:.3f}")
print(f"Train/val gap:   {gap:.3f} cycles (>5 suggests real overfitting)")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Evaluation 1: internal validation split (as before)


In [ ]:
y_pred_val = model.predict(X_val, verbose=1).reshape(-1)

mae_val = mean_absolute_error(y_val, y_pred_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
r2_val = r2_score(y_val, y_pred_val)

print("INTERNAL VALIDATION SPLIT")
print(f"MAE  : {mae_val:.4f} cycles")
print(f"RMSE : {rmse_val:.4f} cycles")
print(f"R2   : {r2_val:.4f}")

## Evaluation 2: low-RUL region only (NEW -- the safety-relevant number)

Global R²/MAE can look great while the model is still inaccurate exactly
in the window that matters: the last ~30 cycles before failure. This is
the metric to watch if the goal is catching failure before it happens --
not the global R².


In [ ]:
LOW_RUL_THRESHOLD = 30

low_rul_mask = y_val <= LOW_RUL_THRESHOLD
n_low = low_rul_mask.sum()

if n_low > 0:
    mae_low = mean_absolute_error(y_val[low_rul_mask], y_pred_val[low_rul_mask])
    rmse_low = np.sqrt(mean_squared_error(y_val[low_rul_mask], y_pred_val[low_rul_mask]))
    print(f"LOW-RUL SEGMENT (RUL <= {LOW_RUL_THRESHOLD}), n = {n_low}")
    print(f"MAE  : {mae_low:.4f} cycles")
    print(f"RMSE : {rmse_low:.4f} cycles")
    print()
    print("If this MAE is much worse than the global MAE above, the model")
    print("is comparatively less reliable close to failure -- the region")
    print("that actually matters for early-warning use, even though it")
    print("barely moves the global R2.")
else:
    print("No validation samples at or below the low-RUL threshold.")

## Evaluation 3: OFFICIAL NASA test set (NEW -- the number to actually trust)

Requires `prepare_official_test_set.py` to have been run first (uses the
scaler fit on training data only, `.transform()`-ed onto the true test
engines -- no re-fitting, no leakage). These 100 engines were never seen
in training or in the internal validation split, and each trace is
truncated at an arbitrary point before failure -- this is the standard
FD001 literature benchmark.


In [ ]:
import os

if os.path.exists(X_TEST_OFFICIAL_PATH) and os.path.exists(Y_TEST_OFFICIAL_PATH):
    X_test_official = np.load(X_TEST_OFFICIAL_PATH)
    y_test_official = np.load(Y_TEST_OFFICIAL_PATH)

    assert X_test_official.shape[2] == number_of_features, (
        "Official test set feature count doesn't match training features -- "
        "re-run prepare_official_test_set.py after any feature change."
    )

    y_pred_official = model.predict(X_test_official, verbose=1).reshape(-1)

    mae_official = mean_absolute_error(y_test_official, y_pred_official)
    rmse_official = np.sqrt(mean_squared_error(y_test_official, y_pred_official))
    r2_official = r2_score(y_test_official, y_pred_official)

    print("OFFICIAL NASA TEST SET (100 unseen engines, truncated traces)")
    print(f"MAE  : {mae_official:.4f} cycles")
    print(f"RMSE : {rmse_official:.4f} cycles")
    print(f"R2   : {r2_official:.4f}")

    low_mask_official = y_test_official <= LOW_RUL_THRESHOLD
    if low_mask_official.sum() > 0:
        mae_official_low = mean_absolute_error(
            y_test_official[low_mask_official], y_pred_official[low_mask_official]
        )
        print(f"\nLow-RUL segment (RUL <= {LOW_RUL_THRESHOLD}), n = {low_mask_official.sum()}")
        print(f"MAE  : {mae_official_low:.4f} cycles")

    print("\nCompare this R2 to the ~0.85-0.92 published FD001 range --")
    print("that comparison is only meaningful against THIS number, not the")
    print("internal validation split, since this is genuinely unseen data.")
else:
    print("Official test set files not found.")
    print("Run prepare_official_test_set.py first, then re-run this cell.")